# use when RAM gets stuck even if all kernels are closed and restarted. These are zombie processes from previous crashed sessions. 

In [ ]:
# ─── EMERGENCY: Kill zombie GPU processes and reclaim VRAM ───────────────────
import subprocess, os, torch

# Step 1: See exactly what's on the GPU
print("=== Current GPU processes ===")
result = subprocess.run(
    ['nvidia-smi', '--query-compute-apps=pid,process_name,used_gpu_memory',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result.stdout)

# Step 2: Kill the known zombie PIDs
ZOMBIE_PIDS = [3851642, 3442264]  # update from nvidia-smi output above if different

for pid in ZOMBIE_PIDS:
    try:
        kill = subprocess.run(['kill', '-9', str(pid)], capture_output=True, text=True)
        print(f"  kill -9 {pid} → returncode={kill.returncode}")
    except Exception as e:
        print(f"  Could not kill {pid}: {e}")

# Step 3: Wait and flush CUDA cache
import time, gc
time.sleep(3)
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Step 4: Report free VRAM
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
free      = total - reserved
print(f"\n=== After cleanup ===")
print(f"  Total VRAM  : {total:.1f} GB")
print(f"  Allocated   : {allocated:.2f} GB")
print(f"  Free (approx): {free:.1f} GB")

# Step 5: Confirm zombies are gone
print("\n=== GPU processes remaining ===")
result2 = subprocess.run(
    ['nvidia-smi', '--query-compute-apps=pid,process_name,used_gpu_memory',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print(result2.stdout or "  (none — GPU is clean ✓)")